In [15]:
# install libraries if not already installed
# pip install pandas pandas_datareader fredapi matplotlib seaborn numpy python-dotenv

import pandas as pd
import pandas_datareader as pdr
from fredapi import Fred
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv


In [16]:

# Load environment variables
load_dotenv()

# Configure FRED API
fred_key = os.getenv('FRED_API_KEY')
if not fred_key:
    raise ValueError("FRED_API_KEY not found in environment variables.")
fred = Fred(api_key=fred_key)

# Create output directory
output_dir = 'data_csv'
os.makedirs(output_dir, exist_ok=True)

In [17]:
# --- Date Configuration ---

# 1. VAR Analysis Data
# Timeframe: 1987 to mid-2007 (pre-crisis)
VAR_START_DATE = '1987-01-01'
VAR_END_DATE = '2007-07-01'

# 2. Visuals & Analysis Data
# Timeframe: 2000 to present (for broader context)
VISUALS_START_DATE = '2000-01-01'
VISUALS_END_DATE = '2025-08-01'

# 3. Specific Series Overrides
HOUSING_STARTS_START = '1959-01-01' # Long-term historical context

# --- Ticker Definitions ---

# National Level Data
national_tickers = {
    # VAR Data Series
    'cs': 'CSUSHPINSA',       # Case-Shiller National Home Price Index
    'cs10': 'SPCS10RSA',      # Case-Shiller 10-City Home Price Index
    'pce': 'PCEPI',           # Consumer Price Index (PCE)
    'ffr': 'FEDFUNDS',        # Federal Funds Rate
    'unemp': 'UNRATE',        # Unemployment Rate
    
    # Visuals Data Series
    'permits': 'PERMIT',      # Building Permits
    'population': 'POPTHM',   # Population (thousands)
    'housing starts': 'HOUST',# Housing Starts
    'recessions': 'USREC',    # Recession Indicator
}

# Metropolitan Level Data
# Structure: City Code -> {Variable -> FRED Ticker}
metropol_tickers = {
    'SF': {'hpi': 'SFXRSA', 'permits': 'SANF806BPPRIVSA', 'population': 'SFCPOP'}, # San Francisco
    'CE': {'hpi': 'CEXRSA', 'permits': 'CLEV439BPPRIVSA', 'population': 'CVLPOP'}, # Cleveland
    'CR': {'hpi': 'CRXRNSA', 'permits': 'CHAR737BPPRIVSA', 'population': 'CGRPOP'}, # Charlotte
    'LV': {'hpi': 'LVXRNSA', 'permits': 'LASV832BPPRIV', 'population': 'LSVPOP'},  # Las Vegas
    'MI': {'hpi': 'MIXRNSA', 'permits': 'MIAM112BPPRIVSA', 'population': 'MIMPOP'} # Miami

}

In [18]:
# --- Data Fetching Helper ---
def fetch_fred_series(ticker, start, end, freq=None):
    try:
        series = fred.get_series(ticker, observation_start=start, observation_end=end)
        if freq:
            series = series.resample(freq).ffill()
            # Ensure full range coverage
            full_range = pd.date_range(start=start, end=end, freq=freq)
            series = series.reindex(full_range, method='ffill')
        return series
    except Exception as e:
        print(f"Error fetching {ticker}: {e}")
        return None


In [19]:

print("Starting Data Download...")
var_data = {}
visuals_data = {}

# 1. Process National Data
print("Processing National Data...")
for name, ticker in national_tickers.items():
    # Determine category and date range
    if name in ['permits', 'population', 'housing starts', 'recessions']:
        # Visuals Category
        start = HOUSING_STARTS_START if name in ['housing starts', 'recessions'] else VISUALS_START_DATE
        series = fetch_fred_series(ticker, start, VISUALS_END_DATE)
        if series is not None:
            visuals_data[f"National_{name}"] = series
            print(f"  Fetched {name} (Visuals)")
    else:
        # VAR Category
        series = fetch_fred_series(ticker, VAR_START_DATE, VAR_END_DATE)
        if series is not None:
            var_data[name] = series
            print(f"  Fetched {name} (VAR)")

# 2. Process Metropolitan Data
print("\nProcessing Metropolitan Data...")
for city, tickers in metropol_tickers.items():
    for var_name, ticker in tickers.items():
        col_name = city.lower() if var_name == 'hpi' else f"{city}_{var_name}"
        
        if var_name in ['permits', 'population']:
            # Visuals Category
            # Population needs resampling to monthly
            freq = 'MS' if var_name == 'population' else None
            series = fetch_fred_series(ticker, VISUALS_START_DATE, VISUALS_END_DATE, freq=freq)
            if series is not None:
                visuals_data[col_name] = series
                print(f"  Fetched {col_name} (Visuals)")
        else:
            # VAR Category
            series = fetch_fred_series(ticker, VAR_START_DATE, VAR_END_DATE)
            if series is not None:
                var_data[col_name] = series
                print(f"  Fetched {col_name} (VAR)")


Starting Data Download...
Processing National Data...
  Fetched cs (VAR)
  Fetched cs (VAR)
  Fetched cs10 (VAR)
  Fetched cs10 (VAR)
  Fetched pce (VAR)
  Fetched pce (VAR)
  Fetched ffr (VAR)
  Fetched ffr (VAR)
  Fetched unemp (VAR)
  Fetched unemp (VAR)
  Fetched permits (Visuals)
  Fetched permits (Visuals)
  Fetched population (Visuals)
  Fetched population (Visuals)
  Fetched housing starts (Visuals)
  Fetched housing starts (Visuals)
  Fetched recessions (Visuals)

Processing Metropolitan Data...
  Fetched recessions (Visuals)

Processing Metropolitan Data...
  Fetched sf (VAR)
  Fetched sf (VAR)
  Fetched SF_permits (Visuals)
  Fetched SF_permits (Visuals)
  Fetched SF_population (Visuals)
  Fetched SF_population (Visuals)
  Fetched ce (VAR)
  Fetched ce (VAR)
  Fetched CE_permits (Visuals)
  Fetched CE_permits (Visuals)
  Fetched CE_population (Visuals)
  Fetched CE_population (Visuals)
  Fetched cr (VAR)
  Fetched cr (VAR)
  Fetched CR_permits (Visuals)
  Fetched CR_permits 

In [20]:

# --- Save Data ---
df_var = pd.DataFrame(var_data)
df_visuals = pd.DataFrame(visuals_data)

# Adjust units if necessary
if 'National_permits' in df_visuals.columns:
    df_visuals['National_permits'] *= 1000


In [21]:

# Save to CSV
var_csv_path = os.path.join(output_dir, 'var_data_1987_2007.csv')
visuals_csv_path = os.path.join(output_dir, 'visuals_data_2000_2007.csv')

df_var.to_csv(var_csv_path)
df_visuals.to_csv(visuals_csv_path)

print(f"\nSaved VAR data to {var_csv_path}")
print(f"Saved Visuals data to {visuals_csv_path}")

display(df_var.head())
display(df_visuals.head())


Saved VAR data to data_csv\var_data_1987_2007.csv
Saved Visuals data to data_csv\visuals_data_2000_2007.csv


,cs,cs10,pce,ffr,unemp,sf,ce,cr,lv,mi
1987-01-01,63.733,63.103814,51.894,6.43,6.6,46.955792,54.057393,63.388189,66.358296,68.499290
1987-02-01,64.132,63.804972,52.076,6.10,6.6,47.302675,54.229218,63.936295,67.030115,68.764210
1987-03-01,64.468,64.425674,52.214,6.13,6.6,47.840213,54.365756,64.171042,67.335161,69.231280
1987-04-01,64.972,65.063393,52.409,6.37,6.3,47.984058,54.155370,64.813123,67.882287,69.198943
1987-05-01,65.547,65.815103,52.542,6.85,6.3,48.305065,54.726192,65.181595,67.898858,69.457298


,National_permits,National_population,National_housing starts,National_recessions,SF_permits,SF_population,CE_permits,CE_population,CR_permits,CR_population,LV_permits,LV_population,MI_permits,MI_population
1959-01-01,NaN,NaN,1657.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-02-01,NaN,NaN,1667.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-03-01,NaN,NaN,1620.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-04-01,NaN,NaN,1590.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1959-05-01,NaN,NaN,1498.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
